# Report Workflow — sources in, a report you can hand in out

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/0Smallcat0/report-workflow/blob/master/docs/quickstart_demo.ipynb)

Your agent writes the document; a **deterministic** layer decides what is
allowed onto the page. Every publishable sentence has to link to material from
the sources you supplied, so an invented number or a fabricated citation is
blocked with the gate and the reason that caught it.

This notebook runs the whole path — parse, author, validate, render — and hands
you a finished `.docx`. **No API key, no model call, offline after install.**
The authoring step is done here by a fixed script standing in for your agent, so
the notebook produces the same document every time.

## 1. Install

The package, its renderer (pandoc), and the example material. About a minute.

In [ ]:
!git clone -q --depth 1 https://github.com/0Smallcat0/report-workflow.git
!pip install -q ./report-workflow
!apt-get -qq install -y pandoc > /dev/null 2>&1
!pandoc --version | head -1

## 2. What goes in

Three ordinary files and one sentence of intent — a measurement table, six
months of one figure, and the notes a colleague would hand you.

In [ ]:
!head -3 report-workflow/examples/data/pilot_results.csv
!echo '---'
!head -3 report-workflow/examples/data/monthly_medians.csv
!echo '---'
!head -8 report-workflow/examples/data/pilot_brief.md

## 3. Run it

`prepare` builds the evidence ledger, the scripted author writes claims and
prose against it, then `validate` + `render` check every claim and only then
produce the document.

In [ ]:
!python report-workflow/examples/source_to_report.py --output /content/run

## 4. Why each sentence was allowed

The document is the deliverable; this note is the reason you can defend it.
Every claim, its verdict, and the source row it rests on.

In [ ]:
import glob
from pathlib import Path

docx = glob.glob('/content/run/*/published/report.docx')[0]
note = glob.glob('/content/run/*/published/traceability/client_readable_qa_note.md')[0]

print(Path(note).read_text(encoding='utf-8'))
print('\nDocument:', docx, '-', Path(docx).stat().st_size, 'bytes')

## 5. Take the document

Table of contents, page numbers, a real Word table, and a chart drawn from the CSV.

In [ ]:
try:
    from google.colab import files
    files.download(docx)
except ImportError:
    print('Not running in Colab. The file is at', docx)

## The gate on its own

No pipeline and no schema — a pure function of `(answer, sources)`. This is what
runs inside every report, exposed for use in a test or a RAG pipeline.

In [ ]:
from report_workflow import verify

SOURCES = {
    "time": "Median processing time was 12.4 minutes for the manual baseline and 7.8 minutes with the structured workflow.",
    "error": "The error rate fell from 3.5% to 1.2% under the structured workflow.",
}

honest = verify("The structured workflow cut median processing time to 7.8 minutes.", SOURCES)
print('honest   ->', honest['publishable'])

hallucinated = verify(
    "The structured workflow drove the error rate down to just 0.2% [error]. "
    "An independent audit confirmed the result [audit].",
    SOURCES,
)
print('invented ->', hallucinated['publishable'])
for sentence in hallucinated['sentence_results']:
    if sentence['status'] == 'blocked':
        print(' ', sentence['checker'], '-', sentence['reason'][:120])

## What you just ran

- **FA** — claim/evidence/sentence linkage; rejects citations to evidence that does not exist.
- **FB** — statistical claims must cite quantitative evidence.
- **FE** — content overlap; rejects numbers and quotes absent from the source.
- **FD** — wording must match evidence grade; a measured claim cannot rest on hearsay.

Same input, same verdict, every run — zero tokens, no GPU, works in CI.

**Where next**

- Drive it from Claude Code: copy `report-workflow/agent_skill` to `~/.claude/skills/report-workflow`, then ask for the report in your own words.
- Measured catch rates and the honest limits: [docs/EVIDENCE.md](https://github.com/0Smallcat0/report-workflow/blob/master/docs/EVIDENCE.md)
- Profiles, Chinese documents, your own Word template: [docs/OUTPUT.md](https://github.com/0Smallcat0/report-workflow/blob/master/docs/OUTPUT.md)